**Name : Sai Krishna Varshith**

**Student ID: 20097433**

**Dataset: Tesla Car Model Prediction** — [Kaggle: Tesla Car Models Dataset](https://www.kaggle.com/datasets/meeratif/tesla-car-dataset) ([notebook context](https://www.kaggle.com/code/meeratif/tesla-car-models-dataset/input))

---

*Notebook 2 of 2 — follow-up experiments testing custom classification heads on EfficientNetB0. Standalone notebook: the data-loading cells below duplicate the setup from `01_baseline_four_models.ipynb` so this can run independently. See the repo README for the overall summary, and notebook 1 for the four-model baseline comparison this builds on.*

## Setup — data loading (same as Notebook 1)

Identical to the data-loading cells in `01_baseline_four_models.ipynb`, repeated here so this notebook can run on its own in a fresh Colab session.

In [ ]:
import os
import tensorflow as tf


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_dir = '/content/drive/MyDrive/Deep Learning/Tesla Models'

In [ ]:
img_size = (224, 224)
bat_size = 32
epochs = (30,50)
dataset_dir

'/content/drive/MyDrive/Deep Learning/Tesla Models'

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.utils import class_weight
import numpy as np

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
dataset_dir,
validation_split=0.2,
subset="training",
seed=42,
image_size=img_size,
batch_size = bat_size,
label_mode='categorical'
)

Found 1311 files belonging to 4 classes.
Using 1049 files for training.


In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
dataset_dir,
validation_split=0.2,
subset="validation",
seed=42,
image_size=img_size,
batch_size=bat_size,
label_mode='categorical'
)

Found 1311 files belonging to 4 classes.
Using 262 files for validation.


In [ ]:
class_names = train_ds.class_names
print("Detected classes:", class_names)

Detected classes: ['Model_E', 'Model_S', 'Model_X', 'Model_Y']


In [ ]:
data_augmentation = models.Sequential([
       layers.RandomFlip("horizontal"),
       layers.RandomRotation(0.2),
       layers.RandomContrast(0.2),
       layers.RandomZoom(0.15),
       layers.RandomTranslation(0.1, 0.1)
   ])

Data augmentation techniques in computer vision used to artificially expand a dataset and improve model robustness

Why?
Small dataset, Reduces overfitting, Only applied to training data

In [ ]:
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
y_train_labels = np.concatenate([y.numpy().argmax(axis=1) for _, y in train_ds], axis=0)
computed_weights = class_weight.compute_class_weight(
class_weight='balanced',
classes=np.unique(y_train_labels),
y=y_train_labels
)
class_weights_dict = dict(enumerate(computed_weights))
print("Computed Class Weights to balance training:", class_weights_dict)

Computed Class Weights to balance training: {0: np.float64(0.6288968824940048), 1: np.float64(1.4569444444444444), 2: np.float64(1.1255364806866952), 3: np.float64(1.197488584474886)}


---
# Part 2: Extended Experimentation — EfficientNetB0 Custom Head Search

The follow-up notebook tried adding extra hidden layers to EfficientNetB0's classification head, to see if a heavier head could beat the Part 1 EfficientNetB0 result (best val F1 = 0.688, val accuracy = 68.3%, from the fine-tuned Phase 2 run above).

This section reproduces those three experiments, reusing `train_ds`, `val_ds`, and `class_weights_dict` already defined in Part 1 — no need to reload the data. A shared builder/trainer function is used instead of copy-pasting the model code three times, and each run gets its own variable names and checkpoint filename.

### Debugging notes — what was wrong in the original extended notebook

Before cleaning it up, here's what the original `20097433_CAone_extended.ipynb` was actually doing, bug by bug, since these are useful to recognise in your own code later:

1. **Overwritten history.** All three runs reused the variable name `history_phase2`. Every new    `.fit()` call silently threw away the previous run's training history, so there was no way to    compare Run 1 vs Run 2 afterwards. Fixed here by giving each run its own `history_head_*` variable.
2. **Overwritten checkpoint file.** `model.save('/content/phase1_checkpoint.keras')` was called for    both Run 1 and Run 2, using the *same path already used by the original Phase 1 EfficientNetB0    checkpoint in Part 1*. Each save silently overwrote the previous file. Fixed here with a distinct    filename per run.
3. **Run 3 quietly reverted the fine-tuning.** Runs 1 and 2 kept the last 50 layers of the base model    unfrozen (continuing the Phase 2 fine-tuning approach from Part 1). Run 3 rebuilt EfficientNetB0    from scratch and froze the *entire* base again — going back to pure feature extraction — without    any comment explaining the change.
4. **Run 3 dropped the F1 metric it was still trying to monitor.** Run 3's `model.compile(...)` only    tracked `'accuracy'`, but the `ReduceLROnPlateau` callback right after it still monitored    `val_f1_macro`. Running the original notebook confirms this — the saved output shows:
   ```
   UserWarning: Learning rate reduction is conditioned on metric `val_f1_macro` which is not available.
   Available metrics are: accuracy,loss,val_accuracy,val_loss,learning_rate.
   ```
   The LR schedule silently never fired for the rest of that run.
5. **Run 3 dropped `class_weight` from `.fit()`.** Runs 1 and 2 passed `class_weight=class_weights_dict`    (needed because Model_S and Model_X are underrepresented, per the class-weight analysis in Part 1).    Run 3's `.fit()` call left it out, silently reintroducing the class-imbalance problem the rest of the    project deliberately corrected for.
6. **`keras-tuner` was installed and imported but never used.** No `kt.Hyperband` / `kt.RandomSearch`    call appears anywhere — it looks like an automated hyperparameter search was planned but not    finished. Kept below as a marked TODO rather than deleted, since it's a reasonable next step.

None of these are fixed silently below — each run is built through one shared function so the same mistake can't happen independently in three places.

In [ ]:
# TODO (not yet implemented in either original notebook): an automated search over
# dense_units / dropout / learning_rate using keras-tuner would replace the three
# manual runs below with a proper Hyperband or RandomSearch sweep.
!pip install -q keras-tuner
import keras_tuner as kt

In [ ]:
def build_efficientnet_custom_head(
    dense_units=(256, 64),
    dropout_rates=(0.5, 0.3),
    base_trainable=True,
    unfreeze_last_n=50,
    learning_rate=1e-4,
):
    """
    Builds an EfficientNetB0 classifier with a configurable custom head.

    dense_units      -- sizes of the two extra hidden Dense layers before the output layer
    dropout_rates    -- dropout after the first hidden layer, and again right before the output layer
    base_trainable   -- if False, the whole base model stays frozen (pure feature extraction,
                        like Part 1 Phase 1). If True, the last `unfreeze_last_n` layers are
                        unfrozen for fine-tuning, matching the Part 1 Phase 2 approach.
    unfreeze_last_n  -- ignored when base_trainable=False
    learning_rate    -- Adam learning rate
    """
    base = tf.keras.applications.EfficientNetB0(
        weights='imagenet', include_top=False, input_shape=(224, 224, 3)
    )
    base.trainable = base_trainable
    if base_trainable:
        fine_tune_at = len(base.layers) - unfreeze_last_n
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False
        for layer in base.layers[fine_tune_at:]:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(dense_units[0], activation='relu')(x)
    x = layers.BatchNormalization()(x)   # keeps gradients stable with a deeper head
    x = layers.Dropout(dropout_rates[0])(x)

    x = layers.Dense(dense_units[1], activation='relu')(x)
    x = layers.Dropout(dropout_rates[1])(x)

    outputs = layers.Dense(4, activation='softmax')(x)  # 4 classes
    model = models.Model(inputs, outputs)

    # F1 (macro) is kept in every run so val_f1_macro is always available to
    # ReduceLROnPlateau/EarlyStopping below -- this is what Run 3 got wrong originally.
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.F1Score(average='macro', name='f1_macro')]
    )
    return model, base


def train_custom_head(model, run_name, epochs=80, es_patience=6, lr_patience=4):
    """Trains a custom-head model with early stopping + LR reduction on val_f1_macro,
    using the same class_weights_dict computed in Part 1. Saves to a run-specific checkpoint
    so different runs never overwrite each other or the Part 1 checkpoints."""
    early_stopping = callbacks.EarlyStopping(
        monitor='val_f1_macro', mode='max', patience=es_patience, restore_best_weights=True
    )
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_f1_macro', mode='max', factor=0.5, patience=lr_patience, min_lr=1e-7, verbose=1
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        class_weight=class_weights_dict,
        callbacks=[early_stopping, reduce_lr]
    )
    model.save(f'/content/{run_name}_checkpoint.keras')
    return history

### Run A — small custom head (256 → 64), fine-tuned base
Matches the first experiment in the original extended notebook: two extra Dense layers on top of the GAP output, base model fine-tuned the same way as Part 1 Phase 2 (last 50 layers unfrozen).

In [ ]:
model_head_a, base_head_a = build_efficientnet_custom_head(
    dense_units=(256, 64), dropout_rates=(0.5, 0.3),
    base_trainable=True, unfreeze_last_n=50, learning_rate=1e-4
)
history_head_a = train_custom_head(model_head_a, run_name='phase3_head_a')

### Run B — larger custom head (512 → 128), fine-tuned base
Matches the second experiment: same fine-tuned base as Run A, but a bigger head to test whether more head capacity helps.

In [ ]:
model_head_b, base_head_b = build_efficientnet_custom_head(
    dense_units=(512, 128), dropout_rates=(0.5, 0.4),
    base_trainable=True, unfreeze_last_n=50, learning_rate=1e-4
)
history_head_b = train_custom_head(model_head_b, run_name='phase3_head_b')

### Run C — small head again, base fully frozen, higher learning rate
Matches the third experiment's *intent* (same small head as Run A, but pure feature extraction with a higher learning rate, since the base is frozen). Unlike the original, this version keeps `f1_macro` in the compiled metrics and `class_weight` in the `.fit()` call — see debugging notes 4 and 5 above.

In [ ]:
model_head_c, base_head_c = build_efficientnet_custom_head(
    dense_units=(256, 64), dropout_rates=(0.5, 0.3),
    base_trainable=False, learning_rate=1e-3
)
history_head_c = train_custom_head(model_head_c, run_name='phase3_head_c', epochs=50)

### Comparing the three custom-head runs to the Part 1 EfficientNetB0 baseline

**Note:** the runs above need to actually execute in Colab (GPU + the Tesla Models dataset from Drive) to produce real numbers — that didn't happen here, so the table below is left as a template rather than filled with invented figures. Run the three cells above first, then run the cell below.

In [ ]:
print("=" * 78)
print("           PART 2 — CUSTOM HEAD COMPARISON (vs. Part 1 EfficientNetB0)")
print("=" * 78)
print(f"{'Run':<28} {'Best Val F1':>14} {'Best Val Acc':>14} {'Epochs Run':>12}")
print("-" * 78)

baseline_f1 = max(history_phase2.history['val_f1_macro'])          # Part 1, Section 1, Phase 2
baseline_acc = max(history_phase2.history['val_accuracy'])
print(f"{'Part 1 baseline (GAP head)':<28} {baseline_f1:>14.3f} {baseline_acc*100:>13.1f}% {len(history_phase2.history['accuracy']):>12}")

for run_name, history in [
    ('Run A (256->64, fine-tuned)', history_head_a),
    ('Run B (512->128, fine-tuned)', history_head_b),
    ('Run C (256->64, frozen base)', history_head_c),
]:
    f1 = max(history.history['val_f1_macro'])
    acc = max(history.history['val_accuracy'])
    n_epochs = len(history.history['accuracy'])
    print(f"{run_name:<28} {f1:>14.3f} {acc*100:>13.1f}% {n_epochs:>12}")

print("=" * 78)

### Takeaways to write up

Once the three runs above have actually been executed, worth checking specifically:
- Did either custom head (Run A or B) beat the plain GAP → Dropout → Dense head's 0.688 val F1 from   Part 1? A heavier head *can* help, but it also adds parameters the ~1,049-image training set may   not support — watch the train/val gap the same way Part 1's overfitting analysis does.
- Does Run C (frozen base, higher LR) top out lower than Runs A/B? If so, that supports Part 1's own   conclusion that fine-tuning (unfreezing the last 50 layers) is what drives most of the improvement,   more than head architecture does.
- If none of the three beat the Part 1 baseline, that's a valid and useful finding too — it suggests   this dataset's bottleneck is data quantity/quality rather than model/head capacity, consistent with   the "why we couldn't reach higher accuracy" section in Part 1's summary.